# Surface Code via SLR: Exploration

This notebook explores building surface code memory experiments using SLR (Simple Logical Representation)
and compares the output to our dedicated circuit builder abstraction.

**Goals:**
1. Build a surface code memory experiment using SLR constructs
2. Generate Stim circuit via SLR's `StimGenerator`
3. Compare to our circuit builder's output
4. Identify gaps and potential SLR enhancements needed:
   - TickCircuit generation
   - Detector/observable metadata for DEM generation
   - Guppy code generation

In [ ]:
# Our circuit builder for comparison
from pecos.qec.surface import (
    SurfacePatch,
    generate_guppy_from_patch,
    generate_stim_from_patch,
    generate_tick_circuit_from_patch,
)
from pecos.qec.surface.schedule import compute_cnot_schedule
from pecos.slr import Barrier, Main, QReg
from pecos.slr.gen_codes.gen_stim import StimGenerator
from pecos.slr.gen_codes.guppy import IRGuppyGenerator
from pecos.slr.qeclib.qubit import CX, H, Measure, Prep

## Part 1: Build Surface Code Memory Experiment in SLR

We'll build a d=3 Z-basis memory experiment with 1 syndrome round.

Structure:
1. Allocate data qubits (9 for d=3)
2. For each syndrome round:
   - Allocate ancilla qubits (4 X + 4 Z = 8)
   - H on X ancillas
   - 4 rounds of parallel CX gates
   - H on X ancillas
   - Measure ancillas
3. Measure data qubits

In [ ]:
def build_surface_code_slr(distance: int, num_rounds: int, basis: str) -> Main:
    """Build a surface code memory experiment using SLR.

    Args:
        distance: Code distance (must be odd >= 3)
        num_rounds: Number of syndrome extraction rounds
        basis: 'Z' or 'X' basis

    Returns:
        SLR Main block representing the experiment
    """
    # Get geometry from SurfacePatch
    patch = SurfacePatch.create(distance=distance)
    geom = patch.geometry

    num_data = geom.num_data
    num_x_stab = len(geom.x_stabilizers)
    num_z_stab = len(geom.z_stabilizers)

    # Create registers
    data = QReg("data", num_data)
    x_anc = QReg("ax", num_x_stab)
    z_anc = QReg("az", num_z_stab)

    # Build operations list
    ops = []

    # 1. Prepare data qubits
    for i in range(num_data):
        ops.append(Prep(data[i]))

    # For X-basis, apply H to all data qubits
    if basis.upper() == "X":
        for i in range(num_data):
            ops.append(H(data[i]))

    ops.append(Barrier())

    # 2. Syndrome extraction rounds
    # Get the CNOT schedule
    cnot_rounds = compute_cnot_schedule(patch)

    for _rnd in range(num_rounds):
        # Prepare ancillas
        for i in range(num_x_stab):
            ops.append(Prep(x_anc[i]))
        for i in range(num_z_stab):
            ops.append(Prep(z_anc[i]))

        ops.append(Barrier())

        # H on X ancillas (before CX)
        for i in range(num_x_stab):
            ops.append(H(x_anc[i]))

        ops.append(Barrier())

        # 4 CX rounds
        for cx_round in cnot_rounds:
            for stab_type, stab_idx, data_q in cx_round:
                if stab_type == "X":
                    # X stabilizer: ancilla is control
                    ops.append(CX(x_anc[stab_idx], data[data_q]))
                else:
                    # Z stabilizer: ancilla is target
                    ops.append(CX(data[data_q], z_anc[stab_idx]))
            ops.append(Barrier())

        # H on X ancillas (after CX)
        for i in range(num_x_stab):
            ops.append(H(x_anc[i]))

        ops.append(Barrier())

        # Measure ancillas
        for i in range(num_x_stab):
            ops.append(Measure(x_anc[i]))
        for i in range(num_z_stab):
            ops.append(Measure(z_anc[i]))

        ops.append(Barrier())

    # 3. Final measurement
    # For X-basis, apply H before measurement
    if basis.upper() == "X":
        for i in range(num_data):
            ops.append(H(data[i]))
        ops.append(Barrier())

    for i in range(num_data):
        ops.append(Measure(data[i]))

    # Create the Main block
    return Main(*ops, vargs=[data, x_anc, z_anc])


# Build a d=3, 1-round, Z-basis experiment
slr_prog = build_surface_code_slr(distance=3, num_rounds=1, basis="Z")

print(f"SLR program created with {len(slr_prog.ops)} operations")
print(f"Variables: {[str(v) for v in slr_prog.vars]}")

## Part 2: Generate Stim via SLR's StimGenerator

In [ ]:
# Generate Stim from SLR
stim_gen = StimGenerator()
stim_gen.generate_block(slr_prog)
slr_stim = stim_gen.get_output()

print("=== SLR -> Stim ===")
print(slr_stim)

In [ ]:
# Generate Stim from our circuit builder for comparison
patch = SurfacePatch.create(distance=3)
builder_stim = generate_stim_from_patch(patch, num_rounds=1, basis="Z")

print("=== Circuit Builder -> Stim ===")
print(builder_stim)

In [ ]:
# Compare the two Stim circuits
def count_stim_ops(stim_str: str) -> dict[str, int]:
    """Count operations in a Stim circuit string."""
    counts: dict[str, int] = {}
    for raw_line in stim_str.strip().split("\n"):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        if parts:
            op = parts[0]
            if op not in counts:
                counts[op] = 0
            # Count individual qubits for multi-qubit ops
            if op in ("R", "H", "M"):
                counts[op] += len([p for p in parts[1:] if not p.startswith("(")])
            elif op == "CX":
                counts[op] += len([p for p in parts[1:] if not p.startswith("(")]) // 2
            else:
                counts[op] += 1
    return counts

slr_counts = count_stim_ops(slr_stim)
builder_counts = count_stim_ops(builder_stim)

print("=== Operation Counts Comparison ===")
print(f"{'Operation':<15} {'SLR':<10} {'Builder':<10} {'Match'}")
print("-" * 45)

all_ops = set(slr_counts.keys()) | set(builder_counts.keys())
for op in sorted(all_ops):
    slr_c = slr_counts.get(op, 0)
    builder_c = builder_counts.get(op, 0)
    match = "Yes" if slr_c == builder_c else "NO"
    print(f"{op:<15} {slr_c:<10} {builder_c:<10} {match}")

## Part 3: Attempt Guppy Generation via SLR

Let's see if SLR's Guppy generator can produce code.

In [ ]:
# Try generating Guppy from SLR
try:
    guppy_gen = IRGuppyGenerator()
    guppy_gen.generate_block(slr_prog)
    slr_guppy = guppy_gen.get_output()
    print("=== SLR -> Guppy ===")
    print(slr_guppy[:2000] if len(slr_guppy) > 2000 else slr_guppy)
    if len(slr_guppy) > 2000:
        print(f"... ({len(slr_guppy) - 2000} more characters)")
except (ValueError, TypeError, AttributeError) as e:
    print(f"Guppy generation failed: {e}")
    slr_guppy = None

In [ ]:
# Compare with circuit builder's Guppy output
builder_guppy = generate_guppy_from_patch(patch)

print("=== Circuit Builder -> Guppy (first 1500 chars) ===")
print(builder_guppy[:1500])
print("...")

## Part 4: TickCircuit - Gap Analysis

SLR currently doesn't have a TickCircuit generator. Let's see what would be needed.

In [ ]:
# Our circuit builder produces TickCircuits with rich metadata
import json

tc = generate_tick_circuit_from_patch(patch, num_rounds=1, basis="Z")

print("=== TickCircuit from Circuit Builder ===")
print(f"Ticks: {tc.num_ticks()}")
print(f"Gates: {tc.gate_count()}")
print()

print("Circuit-level metadata:")
print(f"  basis: {tc.get_meta('basis')}")
print(f"  num_measurements: {tc.get_meta('num_measurements')}")
print(f"  num_detectors: {tc.get_meta('num_detectors')}")
print()

print("Tick-level metadata (first 5 ticks):")
for i in range(min(5, tc.num_ticks())):
    phase = tc.get_tick_meta(i, "phase")
    print(f"  Tick {i}: phase={phase}")
print()

print("Detector annotations (first 3):")
detectors = json.loads(tc.get_meta("detectors"))
for det in detectors[:3]:
    print(f"  D{det['id']}: coords={det['coords']}, records={det['records']}")

## Part 5: Gap Analysis Summary

Based on this exploration, here's what SLR would need to match our circuit builder:

In [ ]:
gaps = [
    {
        "feature": "TickCircuit Generator",
        "status": "Missing",
        "description": "SLR has no TickCircuitGenerator. Would need to add one similar to StimGenerator.",
        "effort": "Medium",
    },
    {
        "feature": "Detector Annotations",
        "status": "Missing",
        "description": "SLR has no concept of detectors/observables for DEM generation. "
                       "Would need to add metadata support or a new annotation type.",
        "effort": "Medium-High",
    },
    {
        "feature": "Tick/Phase Metadata",
        "status": "Partial",
        "description": "SLR has Barrier() which maps to TICK, but no semantic phase annotations "
                       "(prep, syndrome, measure). Could add via Comments or new metadata.",
        "effort": "Low",
    },
    {
        "feature": "Gate-level Metadata",
        "status": "Missing",
        "description": "No way to annotate individual gates with labels (e.g., 'sx0', 'Z2').",
        "effort": "Low-Medium",
    },
    {
        "feature": "Stim Generation",
        "status": "Working",
        "description": "StimGenerator produces valid Stim circuits. Gate counts match.",
        "effort": "N/A",
    },
    {
        "feature": "Guppy Generation",
        "status": "Needs Testing",
        "description": "IRGuppyGenerator exists but may need updates for surface code patterns.",
        "effort": "TBD",
    },
]

print("=== SLR Enhancement Gap Analysis ===")
print()
for gap in gaps:
    print(f"Feature: {gap['feature']}")
    print(f"  Status: {gap['status']}")
    print(f"  Description: {gap['description']}")
    print(f"  Effort: {gap['effort']}")
    print()

## Part 6: Potential SLR Enhancements

To make SLR capable of generating the same outputs as our circuit builder, we would need:

### 1. TickCircuitGenerator
```python
class TickCircuitGenerator(Generator):
    """Generate TickCircuit from SLR programs."""
    
    def generate_block(self, block):
        # Convert Barrier() to tick boundaries
        # Handle qubit allocation
        # Preserve gate ordering within ticks
        pass
```

### 2. Detector/Observable Annotations
```python
# Option A: New SLR operation types
class Detector(Node):
    def __init__(self, id, coords, measurements):
        self.id = id
        self.coords = coords
        self.measurements = measurements

# Option B: Metadata on Measure operations
Measure(anc[0], detector_id=0, coords=[0, 0, 0])
```

### 3. Phase/Semantic Annotations
```python
# Could use Comments with structured format
Comment("phase:syndrome_extraction round:0 cx_round:1")

# Or a new annotation type
PhaseAnnotation(phase="cx_round", round=0, cx_round=1)
```

## Part 7: SLR's Surface Code Library

SLR has its own surface code support in `pecos.slr.qeclib.surface`. Let's explore it
and compare to our `pecos.qec.surface.SurfacePatch`.

In [ ]:
# Explore SLR's surface code library
from pecos.slr.qeclib.surface import (
    RotatedSurfacePatch as SLRRotatedPatch,
)
from pecos.slr.qeclib.surface import (
    SurfacePatchOrientation as SLROrientation,
)

# Create an SLR surface patch
slr_patch = SLRRotatedPatch(
    dx=3,
    dz=3,
    orientation=SLROrientation.X_TOP_BOTTOM,
)

print("=== SLR RotatedSurfacePatch ===")
print(f"dx: {slr_patch.dx}")
print(f"dz: {slr_patch.dz}")
print(f"distance: {slr_patch.distance}")
print(f"data qubits: {len(slr_patch.data)}")
print(f"data_reg: {slr_patch.data_reg}")
print(f"stab_gens: {slr_patch.stab_gens[:5]}...")  # First 5 stabilizer generators
print()

# Compare to our SurfacePatch
from pecos.qec.surface import SurfacePatch

our_patch = SurfacePatch.create(distance=3)

print("=== Our SurfacePatch ===")
print(f"dx: {our_patch.dx}")
print(f"dz: {our_patch.dz}")
print(f"distance: {our_patch.distance}")
print(f"num_data: {our_patch.geometry.num_data}")
print(f"x_stabilizers: {len(our_patch.geometry.x_stabilizers)}")
print(f"z_stabilizers: {len(our_patch.geometry.z_stabilizers)}")

In [ ]:
# Compare stabilizer generators
print("=== Stabilizer Comparison ===")
print()
print("SLR stab_gens format: (type, (qubit_indices...))")
for stab in slr_patch.stab_gens:
    print(f"  {stab}")
print()

print("Our patch X stabilizers:")
for s in our_patch.geometry.x_stabilizers:
    print(f"  X{s.index}: qubits={s.data_qubits}")
print()

print("Our patch Z stabilizers:")
for s in our_patch.geometry.z_stabilizers:
    print(f"  Z{s.index}: qubits={s.data_qubits}")

In [ ]:
# Could we use SLR's patch for circuit generation?
# The SLR patch provides:
# - data_reg: QReg for data qubits (already SLR-compatible)
# - stab_gens: stabilizer generators with qubit indices

def build_memory_from_slr_patch(slr_patch, num_rounds: int, basis: str) -> Main:
    """Build memory experiment using SLR's RotatedSurfacePatch.

    This demonstrates using SLR's own surface code representation.
    """
    # Get data register from SLR patch
    data = slr_patch.data_reg
    num_data = len(slr_patch.data)

    # Parse stabilizer generators to get X and Z stabilizers
    x_stabs = [(i, qubits) for i, (stype, qubits) in enumerate(slr_patch.stab_gens) if stype == "X"]
    z_stabs = [(i, qubits) for i, (stype, qubits) in enumerate(slr_patch.stab_gens) if stype == "Z"]

    num_x_stab = len(x_stabs)
    num_z_stab = len(z_stabs)

    print(f"Building from SLR patch: {num_data} data, {num_x_stab} X stabs, {num_z_stab} Z stabs")

    # Create ancilla registers
    x_anc = QReg("ax", num_x_stab)
    z_anc = QReg("az", num_z_stab)

    ops = []

    # Prepare data
    for i in range(num_data):
        ops.append(Prep(data[i]))
    if basis.upper() == "X":
        for i in range(num_data):
            ops.append(H(data[i]))
    ops.append(Barrier())

    # Syndrome rounds (simplified - no parallel scheduling)
    for _rnd in range(num_rounds):
        # Prep ancillas
        for i in range(num_x_stab):
            ops.append(Prep(x_anc[i]))
        for i in range(num_z_stab):
            ops.append(Prep(z_anc[i]))
        ops.append(Barrier())

        # H on X ancillas
        for i in range(num_x_stab):
            ops.append(H(x_anc[i]))
        ops.append(Barrier())

        # CX gates for X stabilizers
        for local_idx, (_, data_qubits) in enumerate(x_stabs):
            for dq in data_qubits:
                ops.append(CX(x_anc[local_idx], data[dq]))

        # CX gates for Z stabilizers
        for local_idx, (_, data_qubits) in enumerate(z_stabs):
            for dq in data_qubits:
                ops.append(CX(data[dq], z_anc[local_idx]))
        ops.append(Barrier())

        # H on X ancillas
        for i in range(num_x_stab):
            ops.append(H(x_anc[i]))
        ops.append(Barrier())

        # Measure ancillas
        for i in range(num_x_stab):
            ops.append(Measure(x_anc[i]))
        for i in range(num_z_stab):
            ops.append(Measure(z_anc[i]))
        ops.append(Barrier())

    # Final measurement
    if basis.upper() == "X":
        for i in range(num_data):
            ops.append(H(data[i]))
        ops.append(Barrier())
    for i in range(num_data):
        ops.append(Measure(data[i]))

    return Main(*ops, vargs=[data, x_anc, z_anc])


# Build using SLR's patch
slr_native_prog = build_memory_from_slr_patch(slr_patch, num_rounds=1, basis="Z")
print(f"\nSLR native program: {len(slr_native_prog.ops)} operations")

### Integration Observations

**SLR's Surface Code Library:**
- Provides `RotatedSurfacePatch` with data register and stabilizer generators
- Stabilizer generators are tuples: `('X', (q0, q1, q2, q3))` or `('Z', (q0, q1, ...))`
- Data register is already an SLR `QReg`, ready for use in SLR programs
- Has visualization support via `RotatedLatticeVisualization`

**Our SurfacePatch:**
- Provides richer geometry: stabilizer objects with indices, logical operators
- Includes CNOT scheduling via `compute_cnot_schedule()`
- Designed for circuit builder integration

**Potential Unification:**
1. SLR's patch could use our geometry computation
2. Our circuit builder could accept SLR's patch format
3. Or: keep them separate but ensure they produce equivalent circuits

## Summary

This exploration shows that:

1. **SLR can build surface code circuits** - The structure is there (Main, QReg, gates, Barrier)

2. **StimGenerator works** - Produces valid Stim circuits with matching gate counts

3. **SLR has its own surface code library** - `pecos.slr.qeclib.surface` provides:
   - `RotatedSurfacePatch` with data registers and stabilizer generators
   - Visualization support
   - But no CNOT scheduling or circuit generation

4. **Key gaps for parity with circuit builder:**
   - No TickCircuit generator
   - No detector/tracked-Pauli annotation support
   - No semantic phase metadata
   - No gate-level metadata (labels, stabilizer info)
   - No CNOT scheduling in SLR's surface library

5. **Potential paths forward:**
   - Keep circuit builder for surface code DEM generation (simpler, purpose-built)
   - Add TickCircuitGenerator to SLR for general use
   - Consider unifying the two surface patch representations
   - Add detector annotations to SLR for DEM support